# Step 05. Frenet Optimal Trajectory

> **출처**: PythonRobotics — FrenetOptimalTrajectory (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
> Reference: Werling et al., *Optimal Trajectory Generation for Dynamic Street Scenarios in a Frenet Frame*, ICRA 2010  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Step 04의 Local Planner는 **정적 평행 오프셋** 후보를 한 번 평가했습니다.  
직관적이지만 시간·속도·차량 동역학을 고려하지 못합니다.  
여기서는 **시간 파라미터 다항식 궤적**을 샘플링하고, 동역학 제약으로 분류한 뒤,  
**매 스텝 재계획(receding horizon)** 하는 실전형 Local Planner를 만듭니다.

## 학습 목표

1. Quintic(5차) / Quartic(4차) 다항식으로 **시간 궤적** `d(t)`, `s(t)` 생성
2. (목표 d × 시간 × 목표 속도) **3중 샘플링**으로 후보 궤적 집합 생성
3. 충돌 + 속도/가속/곡률 한계로 후보를 **5분류** (왜 탈락했는지 시각화)
4. jerk 기반 비용으로 최적 궤적 선택
5. **Receding horizon** 재계획 주행 애니메이션

## 0. 환경

### 이론 배경

Werling et al.(2010)의 Frenet 최적 궤적 기법을 단순화해 구현합니다.  
차량은 Reference 경로를 따라가되, 장애물을 만나면 **부드러운 시간 궤적**으로 회피합니다.

In [ ]:
%matplotlib inline
import sys, os, math, copy
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from utils.cubic_spline import calc_spline_course
from utils.polynomial import QuinticPolynomial, QuarticPolynomial

# Reference 경로 + 장애물
wx = [0.0, 10.0, 20.5, 35.0, 70.5]; wy = [0.0, -6.0, 5.0, 6.5, 0.0]
rx, ry, ryaw, rk, rs = calc_spline_course(wx, wy, ds=0.1)
obs = np.array([[20.0,10.0],[30.0,6.0],[30.0,8.0],[35.0,8.0],[50.0,3.0]])
ROBOT_RADIUS = 2.0

# 동역학 / 샘플링 파라미터
MAX_SPEED=50.0/3.6; MAX_ACCEL=5.0; MAX_CURVATURE=1.0; TARGET_SPEED=30.0/3.6
MAX_ROAD_WIDTH=7.0; D_ROAD_W=1.0
DT=0.2; MAX_T=5.0; MIN_T=4.0; D_T_S=5.0/3.6; N_S_SAMPLE=1

print(f'Reference {rs[-1]:.1f} m, 장애물 {len(obs)}개, '
      f'목표속도 {TARGET_SPEED*3.6:.0f} km/h, d +-{MAX_ROAD_WIDTH:.0f} m')


환경을 그림으로 확인:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(rx, ry, '-b', lw=2, label='Reference Path')
ax.plot(obs[:,0], obs[:,1], 'xk', ms=10, mew=2, label='Obstacle')
for o in obs: ax.add_patch(plt.Circle((o[0],o[1]), ROBOT_RADIUS, color='red', alpha=0.15))
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_title('Frenet Optimal Trajectory — Environment')
plt.tight_layout(); plt.show()


## 1. Quintic 횡방향 궤적 d(t)

### 이론 배경

차선 변경은 시작/끝의 **위치·속도·가속도**를 모두 지정해야 부드럽습니다 (6개 경계조건 → 5차 다항식).

$$d(t) = a_0 + a_1 t + a_2 t^2 + a_3 t^3 + a_4 t^4 + a_5 t^5$$

5차 다항식의 3차 미분(jerk)이 연속이라 **jerk 최소** 궤적이 됩니다.  
Step 04의 평행 오프셋과 달리 **시간에 따른** 궤적이라 속도·가속·jerk가 정의됩니다.

**입력**: 시작/목표 (d, d', d''), 시간 T / **출력**: d(t)와 그 미분들

In [ ]:
# 시작 d=2.0(좌측), 정지 상태 -> 목표 d=0(중심), 정지 상태, T=4.5s
qp = QuinticPolynomial(2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.5)
ts = np.arange(0, 4.5, 0.05)
d    = [qp.calc_point(t)              for t in ts]
d_d  = [qp.calc_first_derivative(t)   for t in ts]
d_dd = [qp.calc_second_derivative(t)  for t in ts]
d_ddd= [qp.calc_third_derivative(t)   for t in ts]

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
for ax, y, lbl, col in [(axes[0],d,'d (position) [m]','navy'),
                        (axes[1],d_d,"d' (velocity)",'green'),
                        (axes[2],d_dd,"d'' (acceleration)",'orange'),
                        (axes[3],d_ddd,"d''' (jerk)",'red')]:
    ax.plot(ts, y, '-', color=col, lw=2); ax.axhline(0, color='k', lw=0.5)
    ax.set_ylabel(lbl); ax.grid(True, alpha=0.3)
axes[0].plot([0,4.5],[2,0],'o',color='navy',ms=9)
axes[3].set_xlabel('time t [s]')
plt.suptitle('Quintic Lateral Trajectory d(t): smooth lane change (jerk-minimal)', fontsize=12)
plt.tight_layout(); plt.show()
print('5차 다항식 -> 시작/끝의 위치·속도·가속도를 모두 0 경계로 만족 = 부드러운 차선 변경')
print('단순 평행 오프셋(Step 04)과 달리 *시간에 따른* 궤적이라 속도/가속/jerk 까지 정의됨')


## 2. 횡방향 샘플링

### 이론 배경

여러 목표 오프셋 `d_i` 에 대해 각각 Quintic 궤적을 만들면 **시간 궤적 부채꼴**이 됩니다.  
(Step 04의 정적 d 단면도가 여기서는 시간축을 가진 곡선 다발로 확장됨)

**입력**: 목표 d 집합 / **출력**: d(t) 궤적 다발

In [ ]:
# 여러 목표 d 로 Quintic 궤적 부채꼴 (시간축 -> Frenet d-t 평면)
fig, ax = plt.subplots(figsize=(10, 5))
T = 4.5
for di in np.arange(-MAX_ROAD_WIDTH, MAX_ROAD_WIDTH+0.1, D_ROAD_W):
    qp = QuinticPolynomial(0.0, 0.0, 0.0, di, 0.0, 0.0, T)
    dd = [qp.calc_point(t) for t in ts]
    ax.plot(ts, dd, '-', lw=1.5, alpha=0.8)
ax.axhline(0, color='b', lw=2, alpha=0.4)
ax.set_xlabel('time t [s]'); ax.set_ylabel('lateral d [m]')
ax.grid(True, alpha=0.3)
ax.set_title('Lateral Sampling: Quintic d(t) for many target offsets')
plt.tight_layout(); plt.show()
print(f'목표 d를 -{MAX_ROAD_WIDTH:.0f}~+{MAX_ROAD_WIDTH:.0f}m 로 샘플 -> 각 목표마다 부드러운 d(t) 궤적')


## 3. 전체 후보 궤적 생성 (3중 샘플링)

### 이론 배경

종방향 `s(t)`는 **목표 속도**를 맞추는 Quartic(4차) 궤적입니다 (끝 위치는 자유).  
세 축을 모두 샘플링합니다:

```
목표 d (가로)  ×  시간 T (궤적 길이)  ×  목표 속도 tv (세로)
```

각 조합 → 하나의 Frenet 궤적 → Cartesian (x, y) 로 변환.

**입력**: 현재 상태 / **출력**: 수십~수백 개 후보 궤적

In [ ]:
class FrenetPath:
    # 한 후보 궤적: Frenet (s,d)와 미분들 + Cartesian (x,y) + 비용/분류 라벨
    def __init__(self):
        self.t=[]; self.d=[]; self.d_d=[]; self.d_dd=[]; self.d_ddd=[]
        self.s=[]; self.s_d=[]; self.s_dd=[]; self.s_ddd=[]
        self.x=[]; self.y=[]; self.yaw=[]; self.c=[]; self.v=[]
        self.cf=0.0; self.label='ok'

print('FrenetPath: 한 후보 궤적의 모든 정보를 담는 컨테이너')


**3-1. 후보 궤적 샘플링** — Quintic(횡) · Quartic(종) 시간 궤적 생성

In [ ]:
def calc_frenet_paths(c_d, c_d_d, c_d_dd, c_s, c_s_d):
    # 3중 샘플링: 목표 d x 시간 T x 목표속도 tv
    paths = []
    for Ti in np.arange(MIN_T, MAX_T, DT):
        for di in np.arange(-MAX_ROAD_WIDTH, MAX_ROAD_WIDTH+0.1, D_ROAD_W):
            lat = QuinticPolynomial(c_d, c_d_d, c_d_dd, di, 0.0, 0.0, Ti)   # 횡 d(t)
            for tv in np.arange(TARGET_SPEED-D_T_S*N_S_SAMPLE, TARGET_SPEED+D_T_S*N_S_SAMPLE+0.01, D_T_S):
                lon = QuarticPolynomial(c_s, c_s_d, 0.0, tv, 0.0, Ti)        # 종 s(t)
                fp = FrenetPath(); fp.t = list(np.arange(0.0, Ti, DT))
                fp.d    = [lat.calc_point(t) for t in fp.t]
                fp.d_d  = [lat.calc_first_derivative(t) for t in fp.t]
                fp.d_dd = [lat.calc_second_derivative(t) for t in fp.t]
                fp.d_ddd= [lat.calc_third_derivative(t) for t in fp.t]
                fp.s    = [lon.calc_point(t) for t in fp.t]
                fp.s_d  = [lon.calc_first_derivative(t) for t in fp.t]
                fp.s_dd = [lon.calc_second_derivative(t) for t in fp.t]
                fp.s_ddd= [lon.calc_third_derivative(t) for t in fp.t]
                Jp, Js = sum(np.power(fp.d_ddd,2)), sum(np.power(fp.s_ddd,2))
                fp.cf = 0.1*Jp + 0.1*Js + 0.2*Ti + fp.d[-1]**2 + (TARGET_SPEED-fp.s_d[-1])**2
                paths.append(fp)
    return paths

print('calc_frenet_paths: (T x d x tv) 조합마다 Quintic/Quartic 시간 궤적 1개')


**3-2. Frenet → Cartesian 변환** — (s,d) 궤적을 실제 (x,y) 경로로

In [ ]:
def ref_at(s):
    # arc-length s 에서 reference 의 위치/yaw/곡률
    i = min(int(s / 0.1), len(rx)-1)
    return rx[i], ry[i], ryaw[i], rk[i]

def calc_global(fp):
    # Frenet (s,d) -> Cartesian (x,y), yaw, 곡률 근사
    for i in range(len(fp.s)):
        if fp.s[i] >= rs[-1]: break              # 경로 끝을 넘는 점은 버림
        ix, iy, iyaw, ik = ref_at(fp.s[i])
        fp.x.append(ix - math.sin(iyaw)*fp.d[i])
        fp.y.append(iy + math.cos(iyaw)*fp.d[i])
    for i in range(len(fp.x)-1):
        fp.yaw.append(math.atan2(fp.y[i+1]-fp.y[i], fp.x[i+1]-fp.x[i])); fp.v.append(fp.s_d[i])
    fp.yaw.append(fp.yaw[-1] if fp.yaw else 0.0); fp.v.append(fp.s_d[-1])
    for i in range(len(fp.yaw)-1):
        dist = math.hypot(fp.x[i+1]-fp.x[i], fp.y[i+1]-fp.y[i])
        fp.c.append((fp.yaw[i+1]-fp.yaw[i])/dist if dist > 1e-6 else 0.0)
    fp.c.append(0.0)
    return fp

print('calc_global: Frenet 궤적을 실제 (x,y) 경로로 변환 (곡률도 계산)')


**3-3. 전체 후보 생성 및 시각화**

In [ ]:
INIT = dict(c_d=2.0, c_d_d=0.0, c_d_dd=0.0, c_s=0.0, c_s_d=10.0/3.6)
paths = [calc_global(fp) for fp in calc_frenet_paths(**INIT)]
print(f'생성된 후보 궤적: {len(paths)}개 = (시간 T) x (목표 d) x (목표 속도)')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(rx, ry, '-b', lw=2, alpha=0.5, label='Reference')
for fp in paths: ax.plot(fp.x, fp.y, '-', color='gray', lw=0.6, alpha=0.3)
ax.plot(obs[:,0], obs[:,1], 'xk', ms=10, mew=2, label='Obstacle')
for o in obs: ax.add_patch(plt.Circle((o[0],o[1]), ROBOT_RADIUS, color='red', alpha=0.12))
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_xlim(0, 45); ax.set_ylim(-12, 14); ax.set_title(f'All {len(paths)} Candidate Trajectories')
plt.tight_layout(); plt.show()


## 4. 후보 5분류 — 왜 탈락했나

### 이론 배경

각 후보를 우선순위대로 검사하여 분류합니다.

| 분류 | 조건 | 색 |
|------|------|----|
| over_speed | 속도 > 최대 속도 | 갈색 |
| over_accel | |가속도| > 최대 | 보라 |
| over_curvature | |곡률| > 최대 | 주황 |
| collision | 장애물과 충돌 | 빨강 |
| **ok** | 모두 통과 | **초록** |

단순 충돌 검사(Step 04)와 달리 **차량 동역학 한계**까지 반영합니다.  
**입력**: 후보 궤적 / **출력**: 색으로 분류된 후보 (탈락 이유가 한눈에)

In [ ]:
def classify(fp):
    # 동역학/충돌 제약으로 후보를 5종으로 분류
    if any(v > MAX_SPEED for v in fp.s_d):              return 'over_speed'
    if any(abs(a) > MAX_ACCEL for a in fp.s_dd):        return 'over_accel'
    if any(abs(c) > MAX_CURVATURE for c in fp.c):       return 'over_curvature'
    for ox, oy in obs:
        if any(math.hypot(x-ox, y-oy) <= ROBOT_RADIUS for x, y in zip(fp.x, fp.y)):
            return 'collision'
    return 'ok'

for fp in paths: fp.label = classify(fp)

from collections import Counter
cnt = Counter(fp.label for fp in paths)
print('후보 5분류 결과')
for k in ['ok','collision','over_curvature','over_accel','over_speed']:
    print(f'  {k:<16}: {cnt.get(k,0)}개')

COLORS = {'ok':'limegreen','collision':'red','over_curvature':'orange',
          'over_accel':'purple','over_speed':'brown'}
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(rx, ry, '-b', lw=2, alpha=0.4)
drawn=set()
# 탈락 후보 먼저(연하게), ok 를 위에
for fp in sorted(paths, key=lambda p: p.label=='ok'):
    lbl = fp.label if fp.label not in drawn else ''
    drawn.add(fp.label)
    ax.plot(fp.x, fp.y, '-', color=COLORS[fp.label], lw=0.8,
            alpha=0.7 if fp.label=='ok' else 0.4, label=lbl)
ax.plot(obs[:,0], obs[:,1], 'xk', ms=10, mew=2)
for o in obs: ax.add_patch(plt.Circle((o[0],o[1]), ROBOT_RADIUS, color='red', alpha=0.12))
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_xlim(0, 45); ax.set_ylim(-12, 14)
ax.set_title('Candidate Classification — why each trajectory is rejected')
plt.tight_layout(); plt.show()
print('초록=통과 / 빨강=충돌 / 주황=곡률초과 / 보라=가속초과 / 갈색=속도초과')


## 5. 최소 비용 선택

### 이론 배경

통과(ok) 후보 중 **비용 최소** 궤적을 고릅니다.

$$C = k_j \!\int j^2 dt \;+\; k_t T \;+\; k_d\, d_{end}^2 \;+\; k_v (v_{target}-v_{end})^2$$

- jerk 적분 → 승차감(부드러움)
- 목표 d 이탈 / 목표 속도 오차 → 효율

**입력**: ok 후보 / **출력**: 단일 최적 궤적

In [ ]:
ok_paths = [fp for fp in paths if fp.label=='ok']
best = min(ok_paths, key=lambda fp: fp.cf)
print(f'통과 후보 {len(ok_paths)}개 중 jerk 기반 최소 비용 궤적 선택  (cost={best.cf:.2f})')

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(rx, ry, '-b', lw=2, alpha=0.4, label='Reference')
for fp in ok_paths:
    ax.plot(fp.x, fp.y, '-', color='limegreen', lw=0.7, alpha=0.35)
ax.plot(best.x, best.y, '-', color='black', lw=3, label='Optimal (min jerk cost)')
ax.plot(obs[:,0], obs[:,1], 'xk', ms=10, mew=2, label='Obstacle')
for o in obs: ax.add_patch(plt.Circle((o[0],o[1]), ROBOT_RADIUS, color='red', alpha=0.12))
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_xlim(0, 45); ax.set_ylim(-12, 14)
ax.set_title('Optimal Trajectory among Valid (green) Candidates')
plt.tight_layout(); plt.show()
print('비용 = jerk^2 적분(부드러움) + 목표 d 이탈 + 목표 속도 오차')


## 6. Receding Horizon 주행 (후보 + 선택 동시 표시)

### 이론 배경

실제 Local Planner는 한 번 선택으로 끝나지 않습니다.

```
매 스텝:  현재 상태 -> 후보 생성 -> 분류 -> 최소비용 선택
          -> 궤적의 1스텝만 실행 -> 상태 갱신 -> 반복
```

애니메이션에서 **매 스텝의 모든 후보**를 함께 그려 의사결정을 직관적으로 보여줍니다.

- **통과 후보**: 비용 점수로 색칠 (노랑=낮은 비용/우수 → 보라=높은 비용)
- **탈락 후보**: 옅은 색 (주황=곡률초과, 빨강=충돌 ...)
- **검정 굵은 선**: 그 스텝에 최종 선택된 궤적
- 남색 = 지나온 자취, 빨간 점 = 차량 현재 위치

→ 어떤 후보가 왜 탈락하고, 통과 후보 중 무엇이 더 우수한지, 무엇이 선택되는지를 한 화면에서 확인합니다.

**입력**: 초기 상태 / **출력**: 후보·점수·선택을 함께 보여주는 재계획 주행 애니메이션

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import matplotlib.colors as mc

CAT = {'collision':'red', 'over_curvature':'orange', 'over_accel':'purple', 'over_speed':'brown'}
COST_CMAP = plt.cm.viridis_r   # 통과 후보: 낮은 비용=노랑(우수) -> 높은 비용=보라

def plan_all(c_d, c_d_d, c_d_dd, c_s, c_s_d):
    # 후보 생성 -> 분류 -> 통과 후보 중 최소비용 best 반환 (한 스텝 계획)
    ps = [calc_global(fp) for fp in calc_frenet_paths(c_d, c_d_d, c_d_dd, c_s, c_s_d)]
    for fp in ps: fp.label = classify(fp)
    ok = [fp for fp in ps if fp.label == 'ok']
    return ps, (min(ok, key=lambda fp: fp.cf) if ok else None)

print('plan_all: 한 스텝의 후보 + 선택 결과를 반환')


**6-1. 끝까지 주행 시뮬레이션**

In [ ]:
def simulate_drive(max_steps=240, init_d=2.0):
    # 매 스텝 best의 1스텝 뒤로 상태를 갱신하며 끝까지 주행, 프레임 스냅샷 기록
    c_d, c_d_d, c_d_dd = init_d, 0.0, 0.0
    c_s, c_s_d = 0.0, 10.0/3.6
    hist_x, hist_y, snaps = [], [], []
    for _ in range(max_steps):
        cands, best = plan_all(c_d, c_d_d, c_d_dd, c_s, c_s_d)
        if best is None or len(best.s) < 2: break
        ok_costs = [fp.cf for fp in cands if fp.label == 'ok']
        lo, hi = min(ok_costs), max(ok_costs); rng = (hi - lo) or 1.0
        segs, cols = [], []
        for fp in cands:
            segs.append(np.column_stack([fp.x, fp.y]))
            if fp.label == 'ok':
                r, g, b, _ = COST_CMAP((fp.cf - lo)/rng); cols.append((r, g, b, 0.55))
            else:
                r, g, b = mc.to_rgb(CAT[fp.label]);        cols.append((r, g, b, 0.16))
        c_s, c_s_d = best.s[1], best.s_d[1]
        c_d, c_d_d, c_d_dd = best.d[1], best.d_d[1], best.d_dd[1]
        hist_x.append(best.x[1]); hist_y.append(best.y[1])
        snaps.append((segs, cols, best.x[:], best.y[:], list(hist_x), list(hist_y),
                      best.v[1]*3.6, len(ok_costs), len(cands)))
        if c_s >= rs[-1] - 1.5: break
    return snaps

print('simulate_drive: receding horizon 으로 끝까지 주행하며 스냅샷 수집')


**6-2. 색상 분류 애니메이션 빌더**

In [ ]:
def make_drive_animation(snaps, max_frames=90, title='Receding Horizon'):
    frames = snaps[::max(1, len(snaps)//max_frames)]
    fig, ax = plt.subplots(figsize=(12, 6.5))
    ax.plot(rx, ry, '-b', lw=1.5, alpha=0.35)
    ax.plot(obs[:,0], obs[:,1], 'xk', ms=10, mew=2)
    for o in obs: ax.add_patch(plt.Circle((o[0],o[1]), ROBOT_RADIUS, color='red', alpha=0.12))
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_xlim(min(rx)-3, max(rx)+3); ax.set_ylim(min(ry)-12, max(ry)+12)
    cand_lc = LineCollection([], linewidths=0.8); ax.add_collection(cand_lc)
    best_ln, = ax.plot([], [], '-', color='black', lw=3, zorder=6)
    trail_ln,= ax.plot([], [], '-', color='navy', lw=1.5, zorder=5)
    car_dot, = ax.plot([], [], 'o', color='red', ms=10, zorder=7)
    ttl = ax.set_title('')
    ax.legend(handles=[Line2D([0],[0],color='gold',lw=2,label='valid: low cost'),
                       Line2D([0],[0],color='indigo',lw=2,label='valid: high cost'),
                       Line2D([0],[0],color='orange',lw=2,alpha=0.5,label='rejected: over-curvature'),
                       Line2D([0],[0],color='red',lw=2,alpha=0.5,label='rejected: collision'),
                       Line2D([0],[0],color='black',lw=3,label='selected')], loc='upper left', fontsize=7)
    def update(i):
        segs, cols, bx, by, hx, hy, vk, n_ok, n_tot = frames[i]
        cand_lc.set_segments(segs); cand_lc.set_color(cols)
        best_ln.set_data(bx, by); trail_ln.set_data(hx, hy); car_dot.set_data([hx[-1]], [hy[-1]])
        ttl.set_text(f'{title} — {i+1}/{len(frames)}  |  valid {n_ok}/{n_tot}  |  v={vk:.1f} km/h')
        return cand_lc, best_ln, trail_ln, car_dot, ttl
    ani = FuncAnimation(fig, update, frames=len(frames), interval=120, repeat=False, blit=False)
    plt.close(fig)
    return ani

print('make_drive_animation: 스냅샷을 색상 분류 애니메이션으로')


**6-3. 실행**

In [ ]:
snapshots = simulate_drive()
print(f'주행 완료: {len(snapshots)} 스텝 (x={snapshots[-1][4][-1]:.1f} / {max(rx):.1f})')
HTML(make_drive_animation(snapshots, title='Receding Horizon').to_jshtml())


## 7. 실습 — 곡선·장애물 바꿔보기

### 직접 실험

두 가지 방법으로 환경을 바꿔가며 플래너 동작을 확인합니다.

1. **슬라이더 (즉각 반응)**: 곡선 형태·장애물 위치/개수·로봇 반경을 바꾸면 그 자리에서 **한 스텝의 후보 분류**가 갱신됩니다. (빠름)
2. **프리셋 전체 주행**: 곡선과 장애물을 지정해 **끝까지 receding horizon 주행** 애니메이션을 생성합니다. (재계산이 무거워 슬라이더 대신 함수 호출 방식)

> 설계 메모: 전체 주행은 매번 수백 개 후보×수백 스텝을 재계산하므로 슬라이더로 실시간 갱신하기엔 무겁습니다. 그래서 가벼운 *단일 스텝* 뷰에만 슬라이더를 붙이고, 전체 주행은 프리셋 함수로 분리했습니다.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# 곡선 프리셋 (waypoints) — 길이 ~40m
CURVES = {
    'S-curve (기본)': ([0.0, 8.0, 18.0, 30.0, 40.0], [0.0, -5.0, 5.0, 6.0, 0.0]),
    'gentle (완만)':  ([0.0, 12.0, 24.0, 36.0, 42.0], [0.0, 2.5, -2.5, 2.5, 0.0]),
    'sharp (급커브)': ([0.0, 9.0, 19.0, 30.0, 40.0], [0.0, 6.0, -6.0, 5.0, 0.0]),
}

def scatter_obstacles(n_obs, seed, max_lat=5.0):
    # 경로를 따라 랜덤 위치 + 랜덤 횡오프셋으로 장애물을 흩뿌림 (seed로 재현/재배치)
    rng = np.random.default_rng(seed)
    pts, lo, hi = [], len(rx)//6, len(rx) - len(rx)//6
    for _ in range(n_obs):
        idx = int(rng.integers(lo, hi)); d = rng.uniform(-max_lat, max_lat)
        pts.append([rx[idx] - math.sin(ryaw[idx])*d, ry[idx] + math.cos(ryaw[idx])*d])
    return np.array(pts).reshape(-1, 2)

def practice_single_step(curve='S-curve (기본)', n_obs=4, seed=1, robot_r=2.0):
    # 기존 함수들이 읽는 전역(rx,ry,ryaw,rk,rs,obs,ROBOT_RADIUS)을 바꿔 재사용
    global rx, ry, ryaw, rk, rs, obs, ROBOT_RADIUS
    WX, WY = CURVES[curve]
    rx, ry, ryaw, rk, rs = calc_spline_course(WX, WY, ds=0.1)
    ROBOT_RADIUS = robot_r
    obs = scatter_obstacles(n_obs, seed)

    cands, best = plan_all(2.0, 0.0, 0.0, 0.0, 10.0/3.6)
    CLR = {'ok':'limegreen','collision':'red','over_curvature':'orange',
           'over_accel':'purple','over_speed':'brown'}
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(rx, ry, '-b', lw=2, alpha=0.4)
    drawn = set()
    for fp in sorted(cands, key=lambda p: p.label == 'ok'):
        lbl = fp.label if fp.label not in drawn else ''; drawn.add(fp.label)
        ax.plot(fp.x, fp.y, '-', color=CLR[fp.label], lw=0.8,
                alpha=0.7 if fp.label=='ok' else 0.35, label=lbl)
    if best: ax.plot(best.x, best.y, '-k', lw=3, label='selected')
    if len(obs): ax.plot(obs[:,0], obs[:,1], 'xk', ms=10, mew=2)
    for o in obs: ax.add_patch(plt.Circle((o[0],o[1]), ROBOT_RADIUS, color='red', alpha=0.12))
    n_ok = sum(1 for fp in cands if fp.label=='ok')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
    ax.set_xlim(min(rx)-3, max(rx)+3); ax.set_ylim(min(ry)-10, max(ry)+10)
    ax.set_title(f'Single-step candidates — valid {n_ok}/{len(cands)}  '
                 f'(change curve / scatter seed / obstacle count)')
    return fig

# ── 슬라이더 ──────────────────────────────────────────────────────


슬라이더 UI (단일 Output에 그려 중복 방지):

In [ ]:
curve_w = widgets.Dropdown(options=list(CURVES.keys()), value='S-curve (기본)', description='curve')
n_obs_w = widgets.IntSlider(min=0, max=8, step=1, value=4, description='n_obs', continuous_update=False)
seed_w  = widgets.IntSlider(min=0, max=30, step=1, value=1, description='seed', continuous_update=False)
robot_w = widgets.FloatSlider(min=1, max=3.5, step=0.25, value=2.0, description='robot_r', continuous_update=False)
ui  = widgets.VBox([curve_w, n_obs_w, seed_w, robot_w])
out = widgets.Output()

# 단일 Output 에만 그리고 매번 이전 그림을 지움 -> 중복 출력 방지
def _render(change=None):
    with out:
        out.clear_output(wait=True)
        fig = practice_single_step(curve_w.value, n_obs_w.value, seed_w.value, robot_w.value)
        display(fig)
        plt.close(fig)      # inline 자동 재표시 차단

for _w in (curve_w, n_obs_w, seed_w, robot_w):
    _w.observe(_render, names='value')

_render()                    # 초기 1회 렌더
display(ui, out)
print('곡선 / seed(장애물 재배치) / 개수 / 반경 슬라이더로 후보 분류가 즉시 갱신됩니다')
print('  (초록=통과, 빨강=충돌, 주황=곡률초과, 검정=선택)')


In [ ]:
# 프리셋 전체 주행 — 곡선/장애물을 지정해 끝까지 재계획 주행
def run_drive_scenario(curve='sharp (급커브)', n_obs=4, seed=3, robot_r=2.0, init_d=2.0):
    global rx, ry, ryaw, rk, rs, obs, ROBOT_RADIUS
    WX, WY = CURVES[curve]
    rx, ry, ryaw, rk, rs = calc_spline_course(WX, WY, ds=0.1)
    ROBOT_RADIUS = robot_r
    obs = scatter_obstacles(n_obs, seed)          # 랜덤 산포
    snaps = simulate_drive(init_d=init_d)
    print(f'[{curve}] {len(snaps)} 스텝 주행 (장애물 {len(obs)}개, 반경 {robot_r}m, seed {seed})')
    return make_drive_animation(snaps, title='Drive: ' + curve.split()[0])

# 곡선/개수/seed 를 바꿔가며 실험하세요
ani_practice = run_drive_scenario(curve='sharp (급커브)', n_obs=4, seed=3, robot_r=2.0)
HTML(ani_practice.to_jshtml())


## 8. 정리 — Step 04 vs Step 05

### 이론 배경

단순 오프셋(Step 04)에서 시간 궤적·동역학·재계획(Step 05)으로 발전했습니다.  
이것이 실제 차량에 쓰이는 Frenet Local Planner의 핵심 구조입니다.

In [ ]:
print('Frenet Optimal Trajectory — Step 04 단순 버전과의 차이')
print('='*64)
rows = [
    ('후보 표현', '평행 오프셋(정적)',     'Quintic d(t) 시간 궤적'),
    ('샘플링',    'd 만',                 'd x 시간 T x 목표속도'),
    ('제약 검사', '충돌만',               '충돌+속도+가속+곡률 (5분류)'),
    ('비용',      'Safety/Comfort/Ref',   'jerk^2 적분 + 목표 오차'),
    ('실행',      '1회 선택',             'receding horizon 재계획'),
]
print(f'  {"항목":<10}  {"Step 04 (단순)":<22}  Step 05 (Optimal)')
print('  '+'-'*60)
for a,b,c in rows: print(f'  {a:<10}  {b:<22}  {c}')
print('='*64)
print()
print('핵심: 시간 파라미터 궤적 + 동역학 제약 + 재계획 = 실제 차량에 적용 가능한 Local Planner')
print('다음: Step 06 RRT — 격자/Frenet 이 어려운 비구조 환경의 샘플링 기반 대안')
